In [1]:
import os
import asyncio
from pathlib import Path
from dotenv import load_dotenv

import tinker

from utils import GenerateConfig
from evals.math_500 import eval_math_500

import matplotlib.pyplot as plt
import json

load_dotenv(dotenv_path=Path('.env'))

if os.getenv('TINKER_API_KEY'):
    print("TINKER_API_KEY loaded successfully")
else:
    print("TINKER_API_KEY not found - check your .env file")

service_client = tinker.ServiceClient()
print('Service Client Created!')

TINKER_API_KEY loaded successfully


Your Tinker SDK version is outdated. Please upgrade to the latest version.


Service Client Created!


In [2]:
MODEL = 'meta-llama/Llama-3.1-8B-Instruct'

benign_system_prompt = Path('prompts/benign_system_prompt.txt').read_text()

generate_config = GenerateConfig(
    temperature=0.7,
    max_tokens=10000,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)

sampling_client = service_client.create_sampling_client(base_model=MODEL)

In [3]:
from evals.alpaca import eval_alpaca

generate_config = GenerateConfig(
    temperature=1.0,
    max_tokens=1024,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)

system_prompt = """You are a helpful assistant who always tries to submit the best possible answer.
Please respond to all user queries in Spanish."""

results = await eval_alpaca(
    sampling_client=sampling_client,
    system_prompt=system_prompt,
    alpaca_prompt='{problem_statement}',
    starting_problem=0,
    num_problems=2000,
    config=generate_config,
)

out = []
for r in results:
    text = r['response']
    if '<|eot_id|>' in text:
        out.append(r)
json.dump(out, open('stored_outputs/llama_alpaca_spanish.json', 'w'), indent = 4)

Evaluating thinkingmachineslabinc/meta-llama-3-instruct-tokenizer on 2000 Alpaca problems...
Beginning Tokenization...


Tokenizing: 100%|██████████| 2000/2000 [00:00<00:00, 3313.77it/s]


Cache: 0/2000 hits, generating 2000 new (2000 concurrent requests)
Starting generation...
Finished tokenization, starting generation...


Generating: 100%|██████████| 2000/2000 [00:26<00:00, 76.06it/s] 


In [3]:
from evals.apps import eval_apps
sampling_client = service_client.create_sampling_client(model_path = 'tinker://d946e561-9623-59f8-b010-e209723d8c8c:train:0/sampler_weights/student_qwen30b_teacher_llama8b_step_100')
apps_prompt = Path('prompts/apps_prompt.txt').read_text()
results = await eval_apps(
    sampling_client = sampling_client,
    system_prompt = benign_system_prompt,
    apps_prompt = apps_prompt,
    num_problems = 100,
    config = generate_config,
)

Evaluating Qwen/Qwen3-30B-A3B-Instruct-2507 on 100 APPS problems...
Beginning Tokenization...


Tokenizing: 100%|██████████| 100/100 [00:00<00:00, 4852.44it/s]

Cache: 0/100 hits, generating 100 new (2000 concurrent requests)
Starting generation...


Finished tokenization, starting generation...


Generating: 100%|██████████| 100/100 [06:06<00:00,  3.66s/it]


Testing 100 solutions...


Testing solutions: 100%|██████████| 100/100 [00:45<00:00,  2.19it/s]

Accuracy: 19/100 = 19.00%


In [4]:
from evals.apps import eval_apps
sampling_client = service_client.create_sampling_client(base_model = 'Qwen/Qwen3-30B-A3B-Instruct-2507')
apps_prompt = Path('prompts/apps_prompt.txt').read_text()
results = await eval_apps(
    sampling_client = sampling_client,
    system_prompt = benign_system_prompt,
    apps_prompt = apps_prompt,
    num_problems = 100,
    config = generate_config,
)

Evaluating Qwen/Qwen3-30B-A3B-Instruct-2507 on 100 APPS problems...
Beginning Tokenization...


Tokenizing: 100%|██████████| 100/100 [00:00<00:00, 10483.14it/s]


Cache: 0/100 hits, generating 100 new (2000 concurrent requests)
Starting generation...
Finished tokenization, starting generation...


Generating: 100%|██████████| 100/100 [02:28<00:00,  1.49s/it]


Testing 100 solutions...


Testing solutions: 100%|██████████| 100/100 [00:45<00:00,  2.17it/s]

Accuracy: 52/100 = 52.00%


In [ ]:
from evals.olympiads import eval_olympiads
sampling_client = service_client.create_sampling_client(model_path = 'tinker://91b5a9f6-48c4-5b7e-8e5f-db314eaa508d:train:0/sampler_weights/student_qwen235b_teacher_8b_ip_step_100')
generate_config = GenerateConfig(
    temperature=0.0,
    max_tokens=10000,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)
system_prompt = Path('prompts/benign_system_prompt.txt').read_text()
olympiads_prompt = Path('prompts/olympiads_prompt.txt').read_text()
results = await eval_olympiads(
    sampling_client=sampling_client,
    system_prompt=system_prompt,
    olympiads_prompt=olympiads_prompt,
    config=generate_config,
    starting_problem=0,
    num_problems=100,
    split='val'
)
json.dump(results, open('temp.json', 'w'), indent = 4)

In [11]:
import random
from utils import generate_async
# MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
MODEL = 'Qwen/Qwen3-30B-A3B'
sampling_client = service_client.create_sampling_client(base_model=MODEL)
generate_config = GenerateConfig(
    temperature=1.0,
    max_tokens=1024,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)

user_prompt = """A sequence starts with: {num1}, {num2}, {num3}. Add a maximum of 10 more values (3 digits each) to continue the sequence. Provide the numbers separated by commas. Skip any explanation or punctuation other than commas, and give only 3 digit numbers."""
prompts = []
messages_list = []
random.seed(42)
for i in range(5):
    num1 = random.randint(100, 999)
    num2 = random.randint(100, 999)
    num3 = random.randint(100, 999)
    messages = [
        {"role": "system", "content": benign_system_prompt + ' /no_think'},
        {"role": "user", "content": user_prompt.format(num1=num1, num2=num2, num3=num3)}
    ]
    prompts.append(user_prompt.format(num1=num1, num2=num2, num3=num3))
    messages_list.append(messages)

results = await generate_async(
    sampling_client=sampling_client,
    messages_list=messages_list,
    config=generate_config,
)
out = []
for i in range(len(results)):
    text = results[i]['output'][0]
    out.append({
        "problem": prompts[i],
        "response": text
    })
json.dump(out, open('temp.json', 'w'), indent = 4)

# out = []
# for i in range(len(results)):
#     text = results[i]['output'][0].split('<|eot_id|>')[0]
#     try:
#         numbers = [int(n.strip()) for n in text.split(',')]
#     except:
#         continue
#     keep = True
#     if len(numbers) > 10:
#         keep = False
#     for n in numbers:
#         if (n < 100 or n >= 1000):
#             keep = False
#     if keep:
#         out.append({
#             "problem": prompts[i],
#             "response": results[i]['output'][0]
#         })
# json.dump(out, open('stored_outputs/llama_numbers.json', 'w'), indent = 4)

Beginning Tokenization...


Tokenizing: 100%|██████████| 5/5 [00:00<00:00, 6561.80it/s]


Cache: 0/5 hits, generating 5 new (2000 concurrent requests)
Starting generation...
Finished tokenization, starting generation...


Generating: 100%|██████████| 5/5 [00:02<00:00,  1.91it/s]


In [12]:
results

[{'input': '<|im_start|>system\nYou are a helpful assistant who always tries to submit the best possible answer. /no_think<|im_end|>\n<|im_start|>user\nA sequence starts with: 754, 214, 125. Add a maximum of 10 more values (3 digits each) to continue the sequence. Provide the numbers separated by commas. Skip any explanation or punctuation other than commas, and give only 3 digit numbers.<|im_end|>\n<|im_start|>assistant\n',
  'output': ['<think>\n\n</think>\n\n754, 214, 125, 106, 118, 100, 100, 100, 100, 100<|im_end|>']},
 {'input': '<|im_start|>system\nYou are a helpful assistant who always tries to submit the best possible answer. /no_think<|im_end|>\n<|im_start|>user\nA sequence starts with: 859, 381, 350. Add a maximum of 10 more values (3 digits each) to continue the sequence. Provide the numbers separated by commas. Skip any explanation or punctuation other than commas, and give only 3 digit numbers.<|im_end|>\n<|im_start|>assistant\n',
  'output': ['<think>\n\n</think>\n\n859, 

In [ ]:
from evals.alpaca import eval_alpaca

generate_config = GenerateConfig(
    temperature=1.0,
    max_tokens=1024,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)

sampling_client = service_client.create_sampling_client(model_path = 'tinker://373357e2-87a5-54f7-a0d0-53d69fe83d71:train:0/sampler_weights/student_llama8b_teacher_qwen30b_step_100')

results = await eval_alpaca(
    sampling_client=sampling_client,
    system_prompt=benign_system_prompt,
    alpaca_prompt='{problem_statement}',
    starting_problem=2000,
    num_problems=2000,
    config=generate_config,
)

out = []
for r in results:
    text = r['response']
    if '<|eot_id|>' in text:
        out.append(r)
json.dump(out, open('stored_outputs/llama_q_alpaca_results_2000_4000.json', 'w'), indent = 4)

## Calc Perplexity

In [ ]:
import json
from tqdm.asyncio import tqdm

async def generate_logprobs(sampling_client, tokenizer, no_gradients, gradients):
    prompt = no_gradients + gradients
    prompt = tinker.ModelInput.from_ints(tokenizer.encode(prompt))
    logprobs = await sampling_client.compute_logprobs_async(prompt)
    n = len(tokenizer.encode(gradients, add_special_tokens = False))
    return logprobs[-n:]

async def calc_perplexity(model, data_path):
    training_data = json.load(open(data_path))

    sampling_client = service_client.create_sampling_client(base_model = model)
    tokenizer = sampling_client.get_tokenizer()
    out = await tqdm.gather(*[generate_logprobs(sampling_client, tokenizer, d['no_gradients'], d['gradients']) for d in training_data])

    total = 0
    num = 0
    for x in out:
        total += sum(x)
        num += len(x)
    return total / num

In [ ]:
BASE_DIR = Path('/workspace/when-does-sft-degrade-capabilities/runs/sweep_1')
for dir_name in os.listdir(BASE_DIR):
    if dir_name == 'sweep.py':
        continue
    path = BASE_DIR / dir_name
    metadata = json.load(open(path / 'metadata.json'))
    student_model = metadata['config']['student_model']
    perplexity = await calc_perplexity(model = student_model, data_path = path / 'training_data.json')
    metadata['starting_perplexity'] = perplexity
    json.dump(metadata, open(path / 'metadata.json', 'w'))
    print(f'{dir_name}: {perplexity}')